# IWM Historical Stock Price Analysis
## Technical Indicators and Trading Signal Generation

This notebook provides an interactive analysis of IWM stock data with various technical indicators and put/call signal generation based on price movement patterns.

In [10]:
# Install required packages
import sys

# Core packages for parquet support and analysis
packages = [
    'pyarrow',           # Parquet file support
    'pandas',            # Data manipulation
    'numpy',             # Numerical computing
    'matplotlib',        # Plotting
    'seaborn'            # Statistical visualization
]

print("Installing required packages...")
for package in packages:
    !{sys.executable} -m pip install -q {package}

print("All packages installed successfully!")

Installing required packages...



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


All packages installed successfully!



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# Install required packages for parquet support
import sys
!{sys.executable} -m pip install -q pyarrow


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# Import the analyzer class from our script
from iwm_analysis import IWMAnalyzer

# Initialize analyzer
analyzer = IWMAnalyzer()

## Step 1: Load Parquet Data

For historical analysis, we load AlphaVantage parquet data directly (up to 5 years of 1-minute bars).

In [13]:
# Load parquet data directly
print("Loading AlphaVantage parquet data...")
df = analyzer.load_parquet_data('IWM', '1min')

# Display basic info
print(f"\nDataFrame shape: {df.shape}")
print(f"\nDate range: {df['Time'].min()} to {df['Time'].max()}")
print(f"\nFirst few rows:")
df.head()

Loading AlphaVantage parquet data...
Loading AlphaVantage parquet data for IWM...
Loaded 1,807,164 rows from parquet
  Including extended hours (4:00 AM - 8:00 PM)
Parquet data loaded: 1,807,164 rows
Date range: 2015-01-02 06:29:00 to 2025-11-14 20:00:00

DataFrame shape: (1807164, 8)

Date range: 2015-01-02 06:29:00 to 2025-11-14 20:00:00

First few rows:


,Time,Open,High,Low,Last,Change,%Chg,Volume
0,2015-01-02 06:29:00,104.0128,104.0128,104.0128,104.0128,0.0000,0.00%,100
1,2015-01-02 06:49:00,104.1777,104.1777,104.1777,104.1777,0.1649,0.16%,100
2,2015-01-02 07:20:00,104.1864,104.1864,104.1864,104.1864,0.0087,0.01%,100
3,2015-01-02 07:30:00,104.2298,104.2819,104.2298,104.2819,0.0955,0.09%,1999
4,2015-01-02 07:32:00,104.1864,104.1864,104.1864,104.1864,-0.0955,-0.09%,300


In [14]:
# Check data quality
print("Missing values per column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nBasic statistics:")
df.describe()

Missing values per column:
Time      0
Open      0
High      0
Low       0
Last      0
Change    0
%Chg      0
Volume    0
dtype: int64

Data types:
Time      datetime64[ns]
Open             float64
High             float64
Low              float64
Last             float64
Change           float64
%Chg              object
Volume             int64
dtype: object

Basic statistics:


,Time,Open,High,Low,Last,Change,Volume
count,1807164,1.807164e+06,1.807164e+06,1.807164e+06,1.807164e+06,1.807164e+06,1.807164e+06
mean,2021-02-07 04:11:19.158847488,1.668175e+02,1.668615e+02,1.667726e+02,1.668174e+02,7.385450e-05,3.906133e+04
min,2015-01-02 06:29:00,8.232300e+01,8.232300e+01,8.225250e+01,8.225250e+01,-3.703240e+01,1.000000e+00
25%,2018-06-19 10:01:45,1.349038e+02,1.349362e+02,1.348715e+02,1.349062e+02,-3.620000e-02,1.033000e+03
50%,2021-07-08 14:14:30,1.701053e+02,1.701537e+02,1.700571e+02,1.701052e+02,0.000000e+00,1.660300e+04
75%,2023-11-13 13:03:15,2.024740e+02,2.025232e+02,2.024150e+02,2.024739e+02,3.670000e-02,4.591125e+04
max,2025-11-14 20:00:00,2.526000e+02,2.527700e+02,2.525200e+02,2.526100e+02,3.703240e+01,8.159656e+06
std,NaN,4.135172e+01,4.136125e+01,4.134126e+01,4.135162e+01,1.754033e-01,9.068954e+04


## Step 2: Calculate Technical Indicators

In [15]:
# Add technical indicators
print("Calculating technical indicators...")
print("This includes:")
print("  - Technical Indicators (ATR, RSI, EMAs, VWAP, RVOL, OBV, StochRSI)")
print("  - Historical Levels (Day, Week, Month, Year)")
print("  - ORB (5-min, 15-min, 30-min Opening Range Breakout)")
print("  - Order Blocks")
print("  Total: 195+ feature columns\n")

df = analyzer.add_technical_indicators(df)

# Display sample of data with indicators
print("\nSample data with indicators:")
indicator_cols = ['Time', 'Last', 'Volume', 'ATR14_W', 'RSI14_W', 'EMA9', 'EMA20', 'EMA50', 
                  'VWAP', 'RVOL20', 'StochRSI_K', 'Prev_Day_High', 'ORB_30m_Trend']
available_cols = [col for col in indicator_cols if col in df.columns]
df[available_cols].tail(20)

Calculating technical indicators...
This includes:
  - Technical Indicators (ATR, RSI, EMAs, VWAP, RVOL, OBV, StochRSI)
  - Historical Levels (Day, Week, Month, Year)
  - ORB (5-min, 15-min, 30-min Opening Range Breakout)
  - Order Blocks
  Total: 195+ feature columns


Calculating technical indicators...
--------------------------------------------------
1/11 - Calculating ATR (Average True Range)...
2/11 - Calculating RSI (Relative Strength Index)...
3/11 - Calculating EMAs (Exponential Moving Averages)...
    - EMA 9...
    - EMA 20...
    - EMA 50...
4/11 - Calculating VWAP (Volume Weighted Average Price)...


KeyboardInterrupt: 

## Step 3: Visualize Price and Indicators

In [ ]:
# Create subplots for visualization
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# Plot 1: Price with EMAs
axes[0].plot(df['Time'], df['Last'], label='Price', color='black', linewidth=1)
axes[0].plot(df['Time'], df['EMA9'], label='EMA9', color='blue', alpha=0.7)
axes[0].plot(df['Time'], df['EMA20'], label='EMA20', color='orange', alpha=0.7)
axes[0].plot(df['Time'], df['EMA50'], label='EMA50', color='red', alpha=0.7)
axes[0].plot(df['Time'], df['VWAP'], label='VWAP', color='purple', alpha=0.7, linestyle='--')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='best')
axes[0].set_title('IWM Price with Moving Averages and VWAP')
axes[0].grid(True, alpha=0.3)

# Plot 2: Volume and RVOL
axes[1].bar(df['Time'], df['Volume'], alpha=0.3, color='gray', label='Volume')
ax1_twin = axes[1].twinx()
ax1_twin.plot(df['Time'], df['RVOL20'], label='RVOL20', color='green', linewidth=2)
ax1_twin.axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Volume')
ax1_twin.set_ylabel('RVOL')
axes[1].legend(loc='upper left')
ax1_twin.legend(loc='upper right')
axes[1].set_title('Volume and Relative Volume')
axes[1].grid(True, alpha=0.3)

# Plot 3: RSI
axes[2].plot(df['Time'], df['RSI14_W'], label='RSI(14)', color='blue', linewidth=2)
axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.5, label='Overbought')
axes[2].axhline(y=30, color='green', linestyle='--', alpha=0.5, label='Oversold')
axes[2].fill_between(df['Time'], 30, 70, alpha=0.1, color='gray')
axes[2].set_ylabel('RSI')
axes[2].set_ylim(0, 100)
axes[2].legend(loc='best')
axes[2].set_title('Relative Strength Index (Wilder)')
axes[2].grid(True, alpha=0.3)

# Plot 4: Stochastic RSI
axes[3].plot(df['Time'], df['StochRSI_K'], label='StochRSI %K', color='blue', linewidth=2)
axes[3].plot(df['Time'], df['StochRSI_D'], label='StochRSI %D', color='red', linewidth=2)
axes[3].axhline(y=80, color='red', linestyle='--', alpha=0.5)
axes[3].axhline(y=20, color='green', linestyle='--', alpha=0.5)
axes[3].fill_between(df['Time'], 20, 80, alpha=0.1, color='gray')
axes[3].set_ylabel('StochRSI')
axes[3].set_ylim(0, 100)
axes[3].legend(loc='best')
axes[3].set_title('Stochastic RSI')
axes[3].set_xlabel('Time')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Generate Trading Signals

In [ ]:
# Generate trading signals
print("Generating technical indicator-based trading signals...")
print("Signal criteria:")
print("  - Consecutive price movements (3+ periods)")
print("  - RSI in appropriate range")
print("  - StochRSI not oversold/overbought")
print("  - Price position relative to VWAP and EMA9")
print("  Minimum 3/5 conditions must be met\n")

signals_df = analyzer.generate_technical_signals(df, consecutive_periods=3)

# Display signal summary
print(f"\nTotal signals: {len(signals_df)}")

if len(signals_df) > 0:
    # Count by signal type
    signal_counts = signals_df['trade_type'].value_counts()
    print("\nSignals by type:")
    for signal_type, count in signal_counts.items():
        print(f"  {signal_type.upper()}: {count}")
    
    # Calculate win rates
    profitable = (signals_df['return_pct'] > 0).sum()
    win_rate = (profitable / len(signals_df)) * 100
    print(f"\nWin Rate: {win_rate:.1f}%")
    print(f"Average Return: {signals_df['return_pct'].mean():.2f}%")
    print(f"Median Duration: {signals_df['duration_minutes'].median():.0f} minutes")
    
    # Show first few signals
    print("\nFirst 10 signals:")
    display_cols = ['trade_type', 'entry_time', 'exit_time', 'entry_price', 'exit_price', 
                   'return_pct', 'duration_minutes', 'signal_strength']
    available_display_cols = [col for col in display_cols if col in signals_df.columns]
    signals_df[available_display_cols].head(10)
else:
    print("No signals generated.")

In [ ]:
# This cell is deprecated - signal generation is now based on technical indicators
# The old run-based analysis has been replaced with a more sophisticated approach
# that uses consecutive price movements combined with technical indicators

print("Note: This notebook now uses technical indicator-based signal generation.")
print("The old run-based analysis has been deprecated.")
print("\nNew signal generation uses:")
print("  - Consecutive price movements")
print("  - RSI levels")
print("  - StochRSI levels")
print("  - Price vs VWAP")
print("  - Price vs EMAs")
print("\nSignals include 117+ columns of context data:")
print("  - 48 Historical Level columns")
print("  - 63 ORB columns (21 per timeframe × 3)")
print("  - 6 Order Block columns")
print("  - Plus standard technical indicators")

## Step 5: Analyze Signal Performance

In [ ]:
# Analyze your actual trading patterns
if os.path.exists('data/trade_patterns.csv'):
    patterns_df = pd.read_csv('data/trade_patterns.csv', index_col=0)
    print("Your Actual Trading Pattern Analysis:")
    print("="*50)
    
    # Group patterns by trade type
    call_patterns = {k: v for k, v in patterns_df.to_dict('index').items() if k.startswith('CALL')}
    put_patterns = {k: v for k, v in patterns_df.to_dict('index').items() if k.startswith('PUT')}
    
    # Display CALL patterns
    if call_patterns:
        print("\nCALL Patterns:")
        for pattern_key, data in call_patterns.items():
            exit_type = pattern_key.replace('CALL_', '')
            print(f"\n  {exit_type}:")
            print(f"    Count: {data['count']:.0f} trades")
            print(f"    Win Rate: {data['profitable_pct']:.1f}%")
            print(f"    Average Return: {data['avg_return']:.3f}%")
            print(f"    Average Duration: {data['avg_duration']:.1f} minutes")
            print(f"    Entry RSI: {data['entry_rsi_mean']:.1f} ± {data['entry_rsi_std']:.1f}")
    
    # Display PUT patterns
    if put_patterns:
        print("\n\nPUT Patterns:")
        for pattern_key, data in put_patterns.items():
            exit_type = pattern_key.replace('PUT_', '')
            print(f"\n  {exit_type}:")
            print(f"    Count: {data['count']:.0f} trades")
            print(f"    Win Rate: {data['profitable_pct']:.1f}%")
            print(f"    Average Return: {data['avg_return']:.3f}%")
            print(f"    Average Duration: {data['avg_duration']:.1f} minutes")
            print(f"    Entry RSI: {data['entry_rsi_mean']:.1f} ± {data['entry_rsi_std']:.1f}")
        
    # Load enriched trades for more detailed analysis
    if os.path.exists('data/trades_enriched.csv'):
        enriched_df = pd.read_csv('data/trades_enriched.csv')
        
        print("\n\nDetailed Pattern Analysis from Your Trades:")
        print("="*50)
        
        for trade_type in ['CALL', 'PUT']:
            type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
            if len(type_data) > 0:
                print(f"\n{trade_type} Summary ({len(type_data)} total scenarios):")
                
                # Overall stats
                print(f"  RSI Range: {type_data['Entry_RSI14_W'].min():.1f} - {type_data['Entry_RSI14_W'].max():.1f}")
                print(f"  Win Rate: {(type_data['Exit_Return'] > 0).mean()*100:.0f}%")
                
                # VWAP position analysis
                below_vwap = (type_data['Entry_Price'] < type_data['Entry_VWAP']).sum()
                print(f"  Below VWAP at entry: {below_vwap/len(type_data)*100:.0f}%")
                
                # Volume analysis
                print(f"  Average RVOL: {type_data['Entry_RVOL20'].mean():.2f}x")
                print(f"  Average ATR: {type_data['Entry_ATR14_W'].mean():.3f}")
else:
    print("No trade patterns found. Run trade_analysis_pipeline.py to analyze your trades.")

In [ ]:
# Visualize your actual trading patterns if data exists
if os.path.exists('data/trades_enriched.csv'):
    enriched_df = pd.read_csv('data/trades_enriched.csv')
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: RSI Distribution by Trade Type
    for trade_type, color in [('CALL', 'green'), ('PUT', 'red')]:
        type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
        if len(type_data) > 0:
            axes[0, 0].hist(type_data['Entry_RSI14_W'], bins=15, alpha=0.6, 
                          label=f'{trade_type} (n={len(type_data)})', color=color, edgecolor='black')
    axes[0, 0].set_xlabel('Entry RSI')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Your RSI Distribution at Entry')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Returns by Exit Type
    exit_returns = []
    exit_labels = []
    for trade_type in ['CALL', 'PUT']:
        for exit_type in ['EXIT', 'STOP_LOSS', 'RUNNER']:
            data = enriched_df[(enriched_df['Trade_Type'] == trade_type) & 
                             (enriched_df['Exit_Type'] == exit_type)]
            if len(data) > 0:
                exit_returns.append(data['Exit_Return'].values)
                exit_labels.append(f'{trade_type}\n{exit_type}')
    
    if exit_returns:
        bp = axes[0, 1].boxplot(exit_returns, labels=exit_labels, patch_artist=True)
        # Color the boxes
        for i, patch in enumerate(bp['boxes']):
            if 'CALL' in exit_labels[i]:
                patch.set_facecolor('lightgreen')
            else:
                patch.set_facecolor('lightcoral')
        axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
        axes[0, 1].set_ylabel('Return (%)')
        axes[0, 1].set_title('Your Returns by Trade Type and Exit')
        axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Duration vs Return Scatter
    for trade_type, color in [('CALL', 'green'), ('PUT', 'red')]:
        type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
        if len(type_data) > 0:
            axes[1, 0].scatter(type_data['Exit_Duration'], type_data['Exit_Return'],
                             alpha=0.6, label=trade_type, color=color, s=50)
    axes[1, 0].set_xlabel('Hold Duration (minutes)')
    axes[1, 0].set_ylabel('Return (%)')
    axes[1, 0].set_title('Your Hold Duration vs Returns')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Plot 4: VWAP Position Analysis
    vwap_data = []
    vwap_labels = []
    for trade_type in ['CALL', 'PUT']:
        type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
        if len(type_data) > 0:
            below = (type_data['Entry_Price'] < type_data['Entry_VWAP']).sum()
            above = len(type_data) - below
            vwap_data.append([below, above])
            vwap_labels.append(trade_type)
    
    if vwap_data:
        x = np.arange(len(vwap_labels))
        width = 0.35
        
        below_counts = [d[0] for d in vwap_data]
        above_counts = [d[1] for d in vwap_data]
        
        axes[1, 1].bar(x - width/2, below_counts, width, label='Below VWAP', color='blue', alpha=0.7)
        axes[1, 1].bar(x + width/2, above_counts, width, label='Above VWAP', color='orange', alpha=0.7)
        
        axes[1, 1].set_ylabel('Number of Trades')
        axes[1, 1].set_title('Your Entry Position Relative to VWAP')
        axes[1, 1].set_xticks(x)
        axes[1, 1].set_xticklabels(vwap_labels)
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.suptitle('Analysis of Your Actual Trading Patterns', y=1.02, fontsize=16)
    plt.show()
else:
    print("No enriched trade data found for visualization.")

## Step 6: Analyze Indicator Values at Entry/Exit

In [ ]:
# Analyze RSI levels at entry for different trade types
print("RSI Analysis at Entry:")
print("="*40)

for trade_type in ['call', 'put']:
    type_df = signals_df[signals_df['trade_type'] == trade_type]
    print(f"\n{trade_type.capitalize()} entries:")
    print(f"Average RSI: {type_df['entry_RSI14_W'].mean():.2f}")
    print(f"Median RSI: {type_df['entry_RSI14_W'].median():.2f}")
    print(f"RSI < 30: {(type_df['entry_RSI14_W'] < 30).sum()} signals")
    print(f"RSI > 70: {(type_df['entry_RSI14_W'] > 70).sum()} signals")

In [ ]:
# Create indicator comparison at entry
indicators_to_compare = ['entry_RSI14_W', 'entry_RVOL20', 'entry_ATR14_W']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, indicator in enumerate(indicators_to_compare):
    for trade_type, color in [('call', 'green'), ('put', 'red')]:
        type_df = signals_df[signals_df['trade_type'] == trade_type]
        axes[idx].hist(type_df[indicator].dropna(), bins=20, alpha=0.5, 
                      label=trade_type, color=color, edgecolor='black')
    
    axes[idx].set_xlabel(indicator.replace('entry_', ''))
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{indicator.replace("entry_", "")} at Entry')
    axes[idx].legend()

plt.tight_layout()
plt.show()

## Step 7: Save Enhanced Data

In [ ]:
# Save results to parquet format (since we loaded parquet data)
print("Saving analysis results...")

# Save enhanced data with indicators
enhanced_file = 'data/iwm_analysis_with_indicators.parquet'
df_save = df.copy()
# Convert %Chg from string to float for parquet
df_save['%Chg'] = df_save['%Chg'].str.rstrip('%').astype(float)
df_save.to_parquet(enhanced_file, index=False)
print(f"Enhanced data saved to: {enhanced_file}")

# Save signals
if len(signals_df) > 0:
    signals_file = 'data/iwm_analysis_signals.parquet'
    signals_df.to_parquet(signals_file, index=False)
    print(f"Signals saved to: {signals_file}")

# Create a summary report
summary = {
    'Data Source': 'AlphaVantage Parquet (Historical)',
    'Data Range': f"{df['Time'].min()} to {df['Time'].max()}",
    'Total Records': len(df),
    'Total Signals': len(signals_df),
    'Call Signals': len(signals_df[signals_df['trade_type'] == 'call']) if len(signals_df) > 0 else 0,
    'Put Signals': len(signals_df[signals_df['trade_type'] == 'put']) if len(signals_df) > 0 else 0,
    'Average Signal Return': f"{signals_df['return_pct'].mean():.3f}%" if len(signals_df) > 0 else 'N/A',
    'Win Rate': f"{(signals_df['return_pct'] > 0).mean()*100:.1f}%" if len(signals_df) > 0 else 'N/A',
}

print("\n" + "="*50)
print("ANALYSIS SUMMARY")
print("="*50)
for key, value in summary.items():
    if isinstance(value, list):
        print(f"{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{key}: {value}")
        
print("\n" + "="*50)
print("FILES CREATED")
print("="*50)
print(f"Enhanced data: {enhanced_file}")
if len(signals_df) > 0:
    print(f"Signals: {signals_file}")